# Multi-objective optimization with Gibbs sampling

This notebook demonstrates how multiple models that implement the `score` operation can be composed for multi-objective sequence generation. The example used here uses Gibbs sampling, but the evedesign framework could also accomodate any other optimization methods such as genetic algorithms.

In [ ]:
import torch
from evedesign.system import System, Protein
from evedesign.tools.mmseqs2 import add_sequences_mmseqs2, filter_entity_sequences_mmseqs
from evedesign.models.evmutation2 import EVmutation2
from evedesign.restraints.seq_dist import LinearSeqDistRestraint
from evedesign.samplers.gibbs import GibbsSampler
from evedesign.types import DeviceType

DEVICE: DeviceType = "cuda" if torch.cuda.is_available() else "cpu"

## Set up system and add sequences

For our example, we use the single-chain chorismate mutase example from our enzyme generation case study.

In [2]:
target_seq = "TSENPLLALREKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDRERDLLERLITLGKAHHLDAHYITRLFQLIIEDSVLTQQALLQQH"

# define single-protein system
system = System([
    Protein(
        id="EcCM", rep=target_seq, first_index=2
    ),
])

In [3]:
# retrieve and add evolutionary sequences to system (this will populate the "sequences" attribute on each entity)
system = add_sequences_mmseqs2(
    system, use_pairing=False, use_env=True
)

# perform some extra redundancy reduction on sequences which ColabFold server does not handle with env=True
system[0].sequences = filter_entity_sequences_mmseqs(
    system[0], max_seq_id=0.90
)

## Set up individual models for multiobjective optimization

We implement a very simple example of multi-objective optimization here which is based on scoring protein fitness and sequence diversity relative to the reference sequence. By adding more scoring models/oracles (e.g. for protein stability, immunogenicity, developability, etc) any biomolecular properties can be jointly optimized.

### First model: evolutionary model

In [4]:
evm2 = EVmutation2(
    encoder_num_samples=4,
    decoder_num_full_samples=8,
    decoder_num_mutant_samples=8,
    device=DEVICE,
    fix_full_decoding_order=True
).build(system)

### Second model: linear sequence distance restraint to target sequence

In [5]:
# create a linear distance restraint relative to reference sequence (first entity with index 0)
dist_restraint = LinearSeqDistRestraint(
    exclude_gaps_from_distance=True
).build(
    system, data={0: [target_seq]}
)

### Gibbs sampling with multi-objective scoring function

Relative weights and signs of scoring methods determine how individual scorers will be incorporated during optimization. E.g. a positive weight on the distance restraint during sampling will enforce sequences to become more dissimilar to the reference sequence, a negative weight will enforce designs to become more similar.

In [6]:
g = GibbsSampler(
    [evm2, dist_restraint],
    weights=[1, -0.1],
    num_sweeps=1,  # to speed up this example, we only use a single sweep, for real applications this needs to be higher
    init_strategy="system",    # initialize sampling with the reference sequence
)

designs = g.generate(
    num_designs=64,
    temperature=0.5
)

2026-03-10 11:41:54.894 | INFO     | evedesign.samplers.gibbs:generate:737 - Gibbs sweep=1/1 step=1/94 T=0.500
2026-03-10 11:42:02.348 | INFO     | evedesign.samplers.gibbs:generate:737 - Gibbs sweep=1/1 step=2/94 T=0.500
2026-03-10 11:42:10.107 | INFO     | evedesign.samplers.gibbs:generate:737 - Gibbs sweep=1/1 step=3/94 T=0.500
2026-03-10 11:42:17.884 | INFO     | evedesign.samplers.gibbs:generate:737 - Gibbs sweep=1/1 step=4/94 T=0.500
2026-03-10 11:42:25.598 | INFO     | evedesign.samplers.gibbs:generate:737 - Gibbs sweep=1/1 step=5/94 T=0.500
2026-03-10 11:42:33.101 | INFO     | evedesign.samplers.gibbs:generate:737 - Gibbs sweep=1/1 step=6/94 T=0.500
2026-03-10 11:42:40.631 | INFO     | evedesign.samplers.gibbs:generate:737 - Gibbs sweep=1/1 step=7/94 T=0.500
2026-03-10 11:42:48.253 | INFO     | evedesign.samplers.gibbs:generate:737 - Gibbs sweep=1/1 step=8/94 T=0.500
2026-03-10 11:42:55.639 | INFO     | evedesign.samplers.gibbs:generate:737 - Gibbs sweep=1/1 step=9/94 T=0.500
2

In [7]:
designs

[SystemInstance([EntityInstance(rep=TSENPLLALRDKISALDEKLLALLAERRELAVEVGKAKLASHRPVRDIDR..., models=None)] id=None score=2.1418121337890623),
 SystemInstance([EntityInstance(rep=TSENPLLALRDKISALDEKLLALLAERRGLAVEVGKAKLASHRPVRDIDR..., models=None)] id=None score=1.9762451171875),
 SystemInstance([EntityInstance(rep=TSENPLLALRDKISALDEKLLALLAERRGLAVEVGKAKLLSHRPVRDIDR..., models=None)] id=None score=1.5305831909179688),
 SystemInstance([EntityInstance(rep=TSENPLLALRDKISALDEKLLALLAERRALAVEVGKAKLLSHRPVRDIDR..., models=None)] id=None score=1.4940994262695313),
 SystemInstance([EntityInstance(rep=TSENPLLALRDKISALDEKLLALLAERRGLAVEVGKAKLASHRPVRDIDR..., models=None)] id=None score=1.9762451171875),
 SystemInstance([EntityInstance(rep=TSENPLLALRDKISALDEKLLALLAERRELAVEVGKAKLLSHRPVRDIDR..., models=None)] id=None score=1.7272018432617187),
 SystemInstance([EntityInstance(rep=TSENPLLALRDKISALDEKLLALLAERRGLAVEVGKAKLASHRPVRDIDR..., models=None)] id=None score=1.9762451171875),
 SystemInstance([EntityInstan

In [ ]:
x